In [101]:
spark

In [102]:
%run /opt/spark-notebooks/silver_base/parsed_data.ipynb


Master DataFrame 'parsed_df' successfully created and listening to Bronze!


In [103]:
from pyspark.sql.functions import col, explode
# 1. Create the database table mapping
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_base.payment_attempt_fact_RT (
        payment_id STRING,
        order_id STRING,
        method STRING,
        provider STRING,
        status STRING,
        failure_reason STRING,
        attempt_time TIMESTAMP,
        source_system STRING,
        bronze_ingest_ts TIMESTAMP
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/base/payment_attempt_fact_RT'
""")

DataFrame[]

In [104]:
payments_exploded_df = parsed_df.select(
    col("data.order.order_id").alias("order_id"),
    explode(col("data.order.payments")).alias("payment"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [105]:
silver_payment_fact_df = payments_exploded_df.select(
    col("payment.payment_id").alias("payment_id"),
    col("order_id"),
    col("payment.method").alias("method"),
    col("payment.provider").alias("provider"),
    col("payment.status").alias("status"),
    col("payment.failure_reason").alias("failure_reason"),
    col("payment.attempt_time").cast("timestamp").alias("attempt_time"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [106]:
payment_fact_query = (
    silver_payment_fact_df.writeStream
    .format("delta")
    .queryName("silver_payment_attempt_fact_stream")
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark-data/checkpoints/silver/base/payment_attempt_fact_RT/v1")
    .trigger(processingTime="30 seconds")
    .start("/opt/spark-data/delta/silver/base/payment_attempt_fact_RT")
)
print("Started silver_payment_attempt_fact_RT continuous stream! ")

Started silver_payment_attempt_fact_RT continuous stream! 


In [109]:
spark.sql("SELECT * FROM silver_base.payment_attempt_fact_RT").limit(20).toPandas()


,payment_id,order_id,method,provider,status,failure_reason,attempt_time,source_system,bronze_ingest_ts
0,PAY-10001,ORD-10001,UPI,PhonePe,SUCCESS,None,2026-01-03 09:06:10,order_service,2026-03-27 16:27:10.891
1,PAY-10002-A,ORD-10002,CARD,MASTERCARD,FAILED,NETWORK_ERROR,2026-01-03 09:21:00,order_service,2026-03-27 16:27:10.891
2,PAY-10002-B,ORD-10002,CARD,MASTERCARD,SUCCESS,None,2026-01-03 09:22:30,order_service,2026-03-27 16:27:10.891
3,PAY-10003,ORD-10003,NET_BANKING,HDFC,SUCCESS,None,2026-01-03 09:42:00,order_service,2026-03-27 16:27:10.891
4,PAY-10005,ORD-10005,UPI,GooglePay,SUCCESS,None,2026-01-03 10:12:00,order_service,2026-03-27 16:27:10.891


In [108]:
spark.sql("SELECT * FROM silver_base.payment_attempt_fact_RT").limit(20).toPandas()


,payment_id,order_id,method,provider,status,failure_reason,attempt_time,source_system,bronze_ingest_ts


In [110]:
for query in spark.streams.active:
    print(query.name)

silver_order_fact_stream
silver_delivery_fact_stream
silver_order_item_fact_stream
bronze_orders_raw_stream
silver_order_event_fact_stream
silver_payment_attempt_fact_stream
silver_discount_fact_stream
landing_raw_events_stream
